# Лабораторная работа №5

**Работа с данными. Расчёт коэффициента корреляции.**

Смысл лабораторной: научиться измерять связь между двумя переменными.

## 0. Импорт библиотек и настройки

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.distributions.empirical_distribution import ECDF

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")
np.random.seed(7)

%matplotlib inline

In [ ]:
def section(title: str) -> None:
    """Печатает заголовок раздела."""
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

## 1. Нормальная выборка x1 (n=100, mean=3, sd=2)

In [ ]:
section("1. Нормальная выборка x1 (n=100, mean=3, sd=2)")

x1 = stats.norm.rvs(loc=3, scale=2, size=100)
print("Первые 10 значений x1:", np.round(x1[:10], 3))

## 2. Выборочные характеристики x1

In [ ]:
section("2. Характеристики x1")

x1_mean = x1.mean()
x1_sd = x1.std(ddof=1)
x1_var = x1.var(ddof=1)

print(f"mean(x1) = {x1_mean:.4f}")
print(f"sd(x1)   = {x1_sd:.4f}")
print(f"var(x1)  = {x1_var:.4f}")

## 3. Биномиальная выборка x2 (size=500, n=10, p=0.5)

In [ ]:
section("3. Биномиальная выборка x2 (size=500, n=10, p=0.5)")

x2 = stats.binom.rvs(n=10, p=0.5, size=500)
print("Первые 20 значений x2:", x2[:20])

## 4. Характеристики x2

In [ ]:
section("4. Характеристики x2")

print(f"mean(x2) = {x2.mean():.4f}")
print(f"sd(x2)   = {x2.std(ddof=1):.4f}")
print(f"var(x2)  = {x2.var(ddof=1):.4f}")
print("\nПолное описание x2:")
print(pd.Series(x2).describe())

## 5-6. ECDF и гистограммы для x1 и x2

In [ ]:
section("5-6. ECDF и гистограммы для x1 и x2")

ecdf_x1 = ECDF(x1)
ecdf_x2 = ECDF(x2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].step(ecdf_x1.x, ecdf_x1.y, where="post",
             color="steelblue", label="x1 (normal)")
axes[0].step(ecdf_x2.x, ecdf_x2.y, where="post",
             color="darkorange", label="x2 (binomial)")
axes[0].set_title("Эмпирические функции распределения (ECDF)")
axes[0].set_xlabel("значение")
axes[0].set_ylabel("F(x)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(x1, bins=15, density=True, alpha=0.6,
             color="steelblue", edgecolor="black", label="x1")
axes[1].hist(x2, bins=range(0, 13), density=True, alpha=0.6,
             color="darkorange", edgecolor="black", align="left", label="x2")
axes[1].set_title("Гистограммы x1 и x2")
axes[1].set_xlabel("значение")
axes[1].set_ylabel("плотность")
axes[1].legend()

plt.tight_layout()
plt.savefig("01_ecdf_hist_x1_x2.png", dpi=120)
plt.show()

## 7. Вектор x2_100: 100 последних значений x2

In [ ]:
section("7. Вектор x2_100 (100 последних значений x2)")

x2_100 = x2[-100:]
print("Длина x2_100:", len(x2_100))
print("Первые 10 значений x2_100:", x2_100[:10])

## 8. Корреляция между x1 и x2_100 (Пирсон, Спирмен, Кендалл)

In [ ]:
section("8. Корреляция между x1 и x2_100")

r_p, p_p = stats.pearsonr(x1, x2_100)
r_s, p_s = stats.spearmanr(x1, x2_100)
r_k, p_k = stats.kendalltau(x1, x2_100)

print(f"Пирсон  : r = {r_p:+.4f}, p = {p_p:.4g}")
print(f"Спирмен : r = {r_s:+.4f}, p = {p_s:.4g}")
print(f"Кендалл : r = {r_k:+.4f}, p = {p_k:.4g}")


def interpret(r: float, p: float, alpha: float = 0.05) -> str:
    """Интерпретация коэффициента корреляции."""
    a = abs(r)
    if a < 0.1:
        strength = "связь отсутствует"
    elif a < 0.5:
        strength = "слабая связь"
    elif a < 0.7:
        strength = "умеренная связь"
    else:
        strength = "сильная связь"
    direction = "прямая" if r > 0 else ("обратная" if r < 0 else "нет")
    sig = "значим" if p < alpha else "НЕ значим"
    return (f"|r|={a:.3f} → {strength}, направление: {direction}; "
            f"p={p:.4g} → коэффициент {sig} (α={alpha})")


print("\nИнтерпретация:")
print("  Пирсон :", interpret(r_p, p_p))
print("  Спирмен:", interpret(r_s, p_s))
print("  Кендалл:", interpret(r_k, p_k))

## 9. График зависимости x1 и x2_100

In [ ]:
section("9. График зависимости x1 и x2_100")

plt.figure(figsize=(7, 5))
plt.scatter(x1, x2_100, alpha=0.7, edgecolors="black", s=45)
plt.xlabel("x1 (нормальное)")
plt.ylabel("x2_100 (биномиальное)")
plt.title(f"Зависимость x1 и x2_100 (Пирсон r = {r_p:+.3f}, p = {p_p:.3g})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("02_scatter_x1_x2_100.png", dpi=120)
plt.show()

## 10. Матрица x3 (объединение x1 и x2_100)

In [ ]:
section("10. Матрица x3 (объединение x1 и x2_100)")

x3 = np.column_stack([x1, x2_100])
print("Форма x3:", x3.shape)
print("Первые 5 строк:\n", np.round(x3[:5], 3))

## 11. Матрица x4 (x2 разбит на 5 столбцов)

In [ ]:
section("11. Матрица x4 (x2 разбит на 5 столбцов)")

n_rows = len(x2) // 5  # 500 // 5 = 100
x4 = x2[: n_rows * 5].reshape(n_rows, 5)
print("Форма x4:", x4.shape)
print("Первые 5 строк:\n", x4[:5])

## 12. Матрица корреляций для x4

In [ ]:
section("12. Матрица корреляций для x4")

corr4 = np.corrcoef(x4, rowvar=False)
print("Матрица корреляций:\n", np.round(corr4, 3))

plt.figure(figsize=(6, 5))
sns.heatmap(corr4, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, center=0,
            xticklabels=[f"c{i+1}" for i in range(5)],
            yticklabels=[f"c{i+1}" for i in range(5)])
plt.title("Матрица корреляций для x4")
plt.tight_layout()
plt.savefig("03_corr_x4.png", dpi=120)
plt.show()

# Проверка значимости парных корреляций
print("\nПарные корреляции с p-value:")
for i in range(5):
    for j in range(i + 1, 5):
        r, p = stats.pearsonr(x4[:, i], x4[:, j])
        print(f"  c{i+1}-c{j+1}: r = {r:+.3f}, p = {p:.3f}")

## 13. Стандартизация x2 (scale и вручную)

In [ ]:
section("13. Стандартизация x2")

# Вручную
x2_st_manual = (x2 - x2.mean()) / x2.std(ddof=1)

# Эквивалент scale() — через numpy (scale использует ddof=1 для sd)
x2_st_scale = (x2 - np.mean(x2)) / np.std(x2, ddof=1)

print("Первые 5 значений (вручную):", np.round(x2_st_manual[:5], 4))
print("Первые 5 значений (scale) :", np.round(x2_st_scale[:5], 4))
print("Совпадают? ", np.allclose(x2_st_manual, x2_st_scale))
print(f"mean(z) ≈ {x2_st_manual.mean():.2e}")
print(f"sd(z)   ≈ {x2_st_manual.std(ddof=1):.4f}")

## 14. log(x1), округление до 3 знаков

In [ ]:
section("14. log(x1), округление до 3 знаков")

# x1 может содержать отрицательные значения (mean=3, sd=2), поэтому
# логарифмируем только положительные значения
x1_positive = x1[x1 > 0]
log_x1 = np.round(np.log(x1_positive), 3)

print(f"Исходных значений: {len(x1)}, положительных: {len(x1_positive)}")
print("Первые 10 значений log(x1):", log_x1[:10])

## 15-18. Датасет mtcars

In [ ]:
section("15-18. Датасет mtcars")


def load_mtcars() -> pd.DataFrame:
    """Загрузка mtcars из нескольких источников с fallback."""
    try:
        import statsmodels.api as sm
        df = sm.datasets.get_rdataset("mtcars", "datasets").data
        return df
    except Exception:
        pass

    url = "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/mtcars.csv"
    try:
        df = pd.read_csv(url)
        return df
    except Exception:
        pass

    raise RuntimeError(
        "Не удалось загрузить mtcars. Установите statsmodels "
        "или обеспечьте доступ к интернету."
    )


mtcars = load_mtcars()
print("Размер mtcars:", mtcars.shape)
print("Колонки:", mtcars.columns.tolist())
print(mtcars.head())

### 15. Точечная диаграмма по mpg

In [ ]:
section("15. Точечная диаграмма mpg")

plt.figure(figsize=(10, 6))
plt.scatter(mtcars["mpg"], range(len(mtcars)),
            c="steelblue", s=60, edgecolors="black")
plt.yticks([])
plt.xlabel("mpg (миль на галлон)")
plt.title("Точечная диаграмма mpg по моделям")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("04_mtcars_mpg_dotchart.png", dpi=120)
plt.show()

### 16. Описательные статистики mpg (вся выборка и по vs)

In [ ]:
section("16. Описательные статистики mpg")

print("=== mpg по всей выборке ===")
print(mtcars["mpg"].describe())

print("\n=== mpg по категориям vs (0 = V-образный, 1 = прямой) ===")
print(mtcars.groupby("vs")["mpg"].describe().round(2))

# t-тест между группами vs=0 и vs=1
g0 = mtcars.loc[mtcars["vs"] == 0, "mpg"]
g1 = mtcars.loc[mtcars["vs"] == 1, "mpg"]
t_stat, p_val = stats.ttest_ind(g0, g1)
print(f"\nt-тест mpg(vs=0) vs mpg(vs=1): t = {t_stat:.3f}, p = {p_val:.4f}")

### 17. ECDF и гистограммы mpg (вся выборка и по vs)

In [ ]:
section("17. ECDF и гистограммы mpg (вся выборка и по vs)")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ec_all = ECDF(mtcars["mpg"])
axes[0, 0].step(ec_all.x, ec_all.y, where="post", color="steelblue")
axes[0, 0].set_title("ECDF mpg (вся выборка)")
axes[0, 0].set_ylabel("F(x)")

axes[0, 1].hist(mtcars["mpg"], bins=10, color="steelblue",
                edgecolor="black", alpha=0.7)
axes[0, 1].set_title("Гистограмма mpg (вся выборка)")
axes[0, 1].set_xlabel("mpg")

for vs_val, color in [(0, "tomato"), (1, "seagreen")]:
    sub = mtcars.loc[mtcars["vs"] == vs_val, "mpg"]
    axes[1, 0].hist(sub, bins=8, alpha=0.6, color=color,
                    edgecolor="black", label=f"vs={vs_val}")
axes[1, 0].set_title("Гистограмма mpg по vs")
axes[1, 0].set_xlabel("mpg")
axes[1, 0].legend()

for vs_val, color in [(0, "tomato"), (1, "seagreen")]:
    sub = mtcars.loc[mtcars["vs"] == vs_val, "mpg"]
    ec_s = ECDF(sub)
    axes[1, 1].step(ec_s.x, ec_s.y, where="post",
                    color=color, label=f"vs={vs_val}")
axes[1, 1].set_title("ECDF mpg по vs")
axes[1, 1].set_xlabel("mpg")
axes[1, 1].set_ylabel("F(x)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("05_mtcars_mpg_ecdf_hist.png", dpi=120)
plt.show()

### 18. log(mpg) и сравнение графиков

In [ ]:
section("18. log(mpg) и сравнение графиков")

mtcars = mtcars.copy()
mtcars["log_mpg"] = np.log(mtcars["mpg"])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ec_log_all = ECDF(mtcars["log_mpg"])
axes[0, 0].step(ec_log_all.x, ec_log_all.y, where="post", color="purple")
axes[0, 0].set_title("ECDF log(mpg) (вся выборка)")
axes[0, 0].set_ylabel("F(x)")

axes[0, 1].hist(mtcars["log_mpg"], bins=10, color="purple",
                edgecolor="black", alpha=0.7)
axes[0, 1].set_title("Гистограмма log(mpg) (вся выборка)")
axes[0, 1].set_xlabel("log(mpg)")

for vs_val, color in [(0, "tomato"), (1, "seagreen")]:
    sub = mtcars.loc[mtcars["vs"] == vs_val, "log_mpg"]
    axes[1, 0].hist(sub, bins=8, alpha=0.6, color=color,
                    edgecolor="black", label=f"vs={vs_val}")
axes[1, 0].set_title("Гистограмма log(mpg) по vs")
axes[1, 0].set_xlabel("log(mpg)")
axes[1, 0].legend()

for vs_val, color in [(0, "tomato"), (1, "seagreen")]:
    sub = mtcars.loc[mtcars["vs"] == vs_val, "log_mpg"]
    ec_s = ECDF(sub)
    axes[1, 1].step(ec_s.x, ec_s.y, where="post",
                    color=color, label=f"vs={vs_val}")
axes[1, 1].set_title("ECDF log(mpg) по vs")
axes[1, 1].set_xlabel("log(mpg)")
axes[1, 1].set_ylabel("F(x)")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("06_mtcars_logmpg_ecdf_hist.png", dpi=120)
plt.show()

print("\nСравнение mpg и log(mpg):")
print(pd.DataFrame({
    "mpg": mtcars["mpg"].describe(),
    "log(mpg)": mtcars["log_mpg"].describe(),
}).round(3))

## Готово!

Все задания выполнены. Графики сохранены в текущей директории:
- `01_ecdf_hist_x1_x2.png`
- `02_scatter_x1_x2_100.png`
- `03_corr_x4.png`
- `04_mtcars_mpg_dotchart.png`
- `05_mtcars_mpg_ecdf_hist.png`
- `06_mtcars_logmpg_ecdf_hist.png`